<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
👉 <b>Note:</b> This Colab notebook illustrates simplified usage of <code>rapidfireai</code>. For the full RapidFire AI experience with advanced experiment manager, UI, and production features, see our <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide.
<br/>
🎬 Watch our <a href="https://youtu.be/nPMBfZWqPWI">intro video</a> to get started!

</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RapidFireAI/rapidfireai/blob/main/community_notebooks/sft-cybersecurity-qa.ipynb)

⚠️ **IMPORTANT:** Do not let the Colab notebook tab stay idle for more than 5 min; Colab will disconnect otherwise. Refresh the TensorBoard screen or interact with the cells to avoid disconnection.

## Fine-Tune a Cybersecurity Q&A Assistant with RapidFire AI

This tutorial demonstrates how to fine-tune an LLM for **cybersecurity question answering** using **Supervised Fine-Tuning (SFT)** with [RapidFire AI](https://github.com/RapidFireAI/rapidfireai), enabling you to train and compare multiple configurations concurrently—even on a single GPU.

## What We’re Building

We’ll fine-tune [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), a 1.5B-parameter instruction-tuned model, to answer cybersecurity questions accurately and concisely. The model will be trained on the [Cybersecurity QA dataset](https://huggingface.co/datasets/mariiazhiv/cybersecurity_qa), which contains instruction-response pairs covering topics like network security, vulnerability analysis, and threat mitigation.

After training, we run a **post-training evaluation** that compares each fine-tuned run against the unmodified baseline using ROUGE-L and BERTScore, producing a final comparison report.

## Our Approach

We use **SFT** with **LoRA (Low-Rank Adaptation)** to efficiently adapt the pre-trained model. To find the best hyperparameters, we compare **4 configurations** simultaneously:

- **2 LoRA adapter sizes**: Lightweight (rank 8, Q/V projections only) vs. Heavy (rank 32, all linear layers including MLP)
- **2 training strategies**: Aggressive (lr=2e-4, linear decay) vs. Stable (lr=5e-5, cosine schedule with warmup)

RapidFire AI’s **chunk-based scheduling** trains all configurations concurrently—splitting the dataset into chunks and letting every run train on each chunk before moving to the next. This provides comparative metrics early and delivers **16–24× faster experimentation throughput** compared to sequential training ([benchmark details](https://huggingface.co/blog/rapidfire-ai-inc/rapidfireai-sft-tutorial)).

### What You’ll Learn

- How to define and run multiple SFT experiments concurrently with RapidFire AI
- Using LoRA adapters of different capacities—including MLP layers—for parameter-efficient fine-tuning
- Building a multiprocessing-safe chat formatter for Qwen models
- Monitoring all training curves simultaneously in TensorBoard
- Running post-training evaluation with ROUGE-L and BERTScore to compare fine-tuned runs against the baseline

## Install RapidFire AI Package and Setup
### Option 1: Install Locally (or on a VM)
For the full RapidFire AI experience—advanced experiment management, UI, and production features—we recommend installing the package on a machine you control (for example, a VM or your local machine) rather than Google Colab. See our [Install and Get Started](https://oss-docs.rapidfire.ai/en/latest/walkthrough.html) guide.

### Option 2: Install in Google Colab
For simplicity, you can run this notebook on Google Colab. This notebook is configured to run end-to-end on Colab with no local installation required. The `bert_score` package is also installed for post-training evaluation.

In [ ]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai==0.14.0  # Takes 1 min
    %pip install bert_score
    !rapidfireai init # Takes 1 min

## Start RapidFire Services

RapidFire AI runs backend services including the **Dispatcher** (REST API on port 8851) and the **Frontend** dashboard (port 8853). The cell below checks whether these services are already running and launches them if not.

- If any issues arise, check the status in a terminal window using `rapidfireai status` or `rapidfireai doctor`.
- You can also run `rapidfireai start` from a terminal on Colab instead of the cell below.

In [ ]:
import subprocess
from time import sleep
import socket
try:
  s = [socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM)]
  s[0].connect(("127.0.0.1", 8851))
  s[1].connect(("127.0.0.1", 8852))
  s[2].connect(("127.0.0.1", 8853))
  s[0].close()
  s[1].close()
  s[2].close()
  print("RapidFire Services are running")
except OSError as error:
  print("RapidFire Services are not running, launching now...")
  subprocess.Popen(["rapidfireai", "start"])
  sleep(30)

## Configure TensorBoard

We use TensorBoard to visualize training loss, evaluation metrics, and learning rate schedules across all 4 runs simultaneously. This lets you compare configurations in real time as they train.

In [ ]:
import os

# Load TensorBoard extension
%load_ext tensorboard

# TensorBoard log directory will be auto-created in experiment path

## Import RapidFire Components

We import the core RapidFire classes:
- **`Experiment`**: Top-level container that groups runs, saves artifacts, and tracks metrics.
- **`RFGridSearch`**: Generates all combinations of your knobs into a Config Group.
- **`RFModelConfig`**: Wraps the base model, LoRA config, and training arguments into a single unit.
- **`RFLoraConfig` / `RFSFTConfig`**: RapidFire wrappers around PEFT’s `LoraConfig` and TRL’s `SFTConfig`, enabling grid-searchable parameters via `List()`.

In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

# NB: If you get "AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" from Colab, just rerun this cell

## Load Dataset

We use the [Cybersecurity QA dataset](https://huggingface.co/datasets/mariiazhiv/cybersecurity_qa), which contains instruction-response pairs covering cybersecurity topics. We also load the Qwen tokenizer here since it’s needed by the chat formatter and generation config below.

Unlike the lighter tutorial notebooks, this notebook uses the **full train and validation splits** rather than a reduced subset.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

dataset = load_dataset("mariiazhiv/cybersecurity_qa")

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

## Define Chat Formatter

SFT training requires a consistent text format. The `QwenChatFormatter` class uses Qwen’s `apply_chat_template` to produce properly formatted chat turns.

This formatter is implemented as a class with lazy tokenizer loading, which is important because RapidFire AI uses multiprocessing: the tokenizer is loaded inside each worker process rather than being pickled across process boundaries.

In [ ]:
class QwenChatFormatter:
    """
    A robust formatter that carries its own tokenizer.
    This works safely across worker processes (multiprocessing).
    """
    def __init__(self, model_id):
        self.model_id = model_id
        self._tokenizer = None # Load lazily

    @property
    def tokenizer(self):
        if self._tokenizer is None:
            from transformers import AutoTokenizer
            # Load locally inside the worker process
            self._tokenizer = AutoTokenizer.from_pretrained(
                self.model_id,
                trust_remote_code=True
            )
            # Ensure pad token exists
            if self._tokenizer.pad_token is None:
                self._tokenizer.pad_token = self._tokenizer.eos_token
        return self._tokenizer

    def __call__(self, example):
        user_input = example.get('input', "") or ""
        instruction = example.get('instruction', "")

        full_content = f"{instruction}\n{user_input}".strip()

        # Create the message structure
        messages = [
            {"role": "user", "content": full_content},
            {"role": "assistant", "content": example['output']}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        return {
            "text": text,
            "user": full_content,
            "assistant": example['output']
        }

# Instantiate the formatter once
qwen_formatter = QwenChatFormatter("Qwen/Qwen2.5-1.5B-Instruct")

## Define Evaluation Metrics

We define a lightweight metrics function using ROUGE-L to measure the quality of generated responses during training. A more thorough post-training evaluation with both ROUGE-L and BERTScore is performed at the end of this notebook.

In [ ]:
def compute_metrics(eval_preds):
    """Lightweight metrics computation"""
    predictions, labels = eval_preds

    try:
        import evaluate

        rouge = evaluate.load("rouge")
        rouge_output = rouge.compute(
            predictions=predictions,
            references=labels,
            use_stemmer=True,
            rouge_types=["rougeL"]
        )

        return {
            "rougeL": round(rouge_output["rougeL"], 4),
        }
    except Exception as e:
        print(f"Metrics computation failed: {e}")
        return {}

## Initialize Experiment

Every RapidFire AI experiment needs a unique name. The `Experiment` object is the top-level container that groups all runs, saves artifacts, and tracks metrics. It automatically creates an MLflow experiment under the hood for structured logging.

In [ ]:
# Create experiment with unique name
my_experiment = "harshit_sft_demo"
experiment = Experiment(experiment_name=my_experiment)

## Get TensorBoard Log Directory

RapidFire AI writes TensorBoard event files to a structured path inside the experiment directory. We retrieve this path so we can point TensorBoard at it for real-time monitoring.

In [ ]:
# Get experiment path
from rapidfireai.fit.db.rf_db import RfDb

db = RfDb()
experiment_path = db.get_experiments_path(my_experiment)
tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{my_experiment}"

print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

## Define Model Configurations

This is where RapidFire AI shines. Instead of hardcoding a single training configuration, we define a search space and let RapidFire run all combinations concurrently.

We use **Qwen2.5-1.5B-Instruct** (1.5B parameters), an instruction-tuned model well-suited for factual Q&A tasks. Two LoRA adapter configs are crossed with two training strategies, producing **4 concurrent runs**.

### Quick LoRA Primer

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning technique that adds small trainable matrices to frozen model layers instead of updating all weights. Key parameters:

- **`r` (rank)**: Dimensionality of the low-rank adapter matrices. We test rank 8 (lightweight) vs. rank 32 (higher capacity).
- **`lora_alpha`**: Scaling factor for LoRA weights, typically set to 2× the rank.
- **`target_modules`**: Which model layers to adapt. We compare adapting only Q/V attention projections vs. all linear layers (attention + MLP gate/up/down projections).
- **`lora_dropout`**: Dropout rate applied to LoRA layers for regularization.

In [ ]:
# Runs:
#   1: Fewer adaptors, aggressive learning
#   2: More adaptors, aggressive learning
#   3: Fewer adaptors, stable learning
#   4: More adaptors, stable learning

peft_configs_qwen = List([
    # Config A: Lightweight - Targets only Query/Value projections
    RFLoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    ),
    # Config B: Heavy - Targets all linear layers (more parameters to learn)
    RFLoraConfig(
        r=32,
        lora_alpha=64,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )
])

config_set_qwen = List([
    # Strategy 1: Aggressive (Higher LR, Linear Decay)
    RFModelConfig(
        model_name=model_id,
        peft_config=peft_configs_qwen,
        training_args=RFSFTConfig(
            learning_rate=2e-4,     # Standard Qwen LoRA rate
            lr_scheduler_type="linear",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=100,           # Short run for the competition demo
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=10,
            fp16=True,              # Required for T4
            gradient_checkpointing=True, # Saves VRAM
            report_to="none",
            num_train_epochs=10,
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "auto",
            "trust_remote_code": True,
            "use_cache": False      # Must be False for Gradient Checkpointing
        },
        formatting_func=qwen_formatter,
        compute_metrics=compute_metrics,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.6,     # Lower temp for factual QA
            "top_p": 0.9,
            "repetition_penalty": 1.1,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
    ),

    # Strategy 2: Stable (Lower LR, Cosine Schedule, Warmup)
    RFModelConfig(
        model_name=model_id,
        peft_config=peft_configs_qwen,
        training_args=RFSFTConfig(
            learning_rate=5e-5,     # Conservative rate
            lr_scheduler_type="cosine",
            warmup_steps=10,        # Gentle start
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=100,
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=10,
            fp16=True,
            gradient_checkpointing=True,
            report_to="none",
            num_train_epochs=10,
        ),
        model_type="causal_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "auto",
            "trust_remote_code": True,
            "use_cache": False
        },
        formatting_func=qwen_formatter,
        compute_metrics=compute_metrics,
        generation_config={
            "max_new_tokens": 256,
            "temperature": 0.6,
            "top_p": 0.9,
            "repetition_penalty": 1.1,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
    )
])

### Define Model Creation Function

RapidFire AI calls this function once per run to instantiate the model and tokenizer from the config dictionary. It receives the expanded config (with the specific LoRA/training args for that run) and must return a `(model, tokenizer)` tuple.

In [ ]:
def create_model(model_config):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"]

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    trust_remote = model_kwargs.get("trust_remote_code", False)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"

    return (model, tokenizer)

### Create Config Group

`RFGridSearch` takes the config set and generates a **Config Group**—the full Cartesian product of all `List()` knobs. Here, 2 LoRA configs × 2 training configs = **4 runs**. Each run in the Config Group will be trained concurrently via chunk-based scheduling.

In [ ]:
config_group = RFGridSearch(
    configs=config_set_qwen,
    trainer_type="SFT"
)

## Start TensorBoard

**IMPORTANT: Make sure to start TensorBoard BEFORE invoking run_fit() below so that you can watch metrics appear in real-time!**

In [ ]:
%tensorboard --logdir {tensorboard_log_dir}

## Run Training + Validation

Now we launch `run_fit()`—the main function for running multi-config training and evaluation. RapidFire AI will:

1. **Expand configs** into 4 individual runs (one per LoRA × strategy combination).
2. **Split the dataset** into `num_chunks=2` chunks for interleaved execution.
3. **Schedule all runs** on each chunk before moving to the next, so you get comparative metrics after the very first chunk.
4. **Log metrics** to TensorBoard in real time—scroll up to watch training loss and eval scores update live.

In [ ]:
experiment.run_fit(
    config_group,
    create_model,
    train_dataset,
    eval_dataset,
    num_chunks=2,
    seed=67
)

## Launch Interactive Run Controller

RapidFire AI provides an Interactive Controller that lets you manage executing runs dynamically in real-time from the notebook:

- ⏹️ **Stop**: Gracefully stop a running config
- ▶️ **Resume**: Resume a stopped run
- 🗑️ **Delete**: Remove a run from this experiment
- 📋 **Clone**: Create a new run by editing the config dictionary of a parent run to try new knob values; optional warm start of parameters
- 🔄 **Refresh**: Update run status and metrics

The Controller uses ipywidgets and is compatible with both Colab (ipywidgets 7.x) and Jupyter (ipywidgets 8.x).

In [ ]:
# Create Interactive Controller
sleep(15)
from rapidfireai.fit.utils.interactive_controller import InteractiveController

controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
controller.display()

## End Experiment

Always end the experiment to finalize logging, save all artifacts, and clean up resources. Click the button below when you’re ready to wrap up.

In [ ]:
from google.colab import output
from IPython.display import display, HTML

display(HTML('''
<button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
'''))

# eval_js blocks until the Promise resolves
output.eval_js('''
new Promise((resolve) => {
    document.getElementById("continue-btn").onclick = () => {
        document.getElementById("continue-btn").disabled = true;
        document.getElementById("continue-btn").innerText = "Continuing...";
        resolve("clicked");
    };
})
''')

# Actually end the experiment after the button is clicked
experiment.end()
print("Done!")

## View TensorBoard Plots and Logs

After your experiment has ended, you can still view the full logs in TensorBoard:

In [ ]:
# View final logs
%tensorboard --logdir {tensorboard_log_dir}

## View RapidFire AI Log Files

RapidFire AI produces structured log files for each experiment. These are useful for debugging, auditing training behavior, or understanding what the scheduler and workers did under the hood.

In [ ]:
# Get the experiment-specific log file
from IPython.display import display, Pretty
log_file = experiment.get_log_file_path()

display(Pretty(f"📄 Experiment Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# Get the training-specific log file
log_file = experiment.get_log_file_path("training")

display(Pretty(f"📄 Training Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

## Post-Training Evaluation: Baseline vs. Fine-Tuned Runs

After training completes, we run a rigorous evaluation that compares all 4 fine-tuned runs against the unmodified **baseline** (the pre-trained Qwen2.5-1.5B-Instruct without any LoRA adapters).

For each run, the evaluation:
1. Loads the base model and applies the saved LoRA adapter checkpoint.
2. Generates answers for the full validation set using batched inference.
3. Computes **ROUGE-L** (longest common subsequence overlap) and **BERTScore** (semantic similarity via DistilBERT embeddings) against reference answers.
4. Produces a final comparison table showing which configuration performed best.

BERTScore runs on CPU to avoid GPU OOM during evaluation.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import evaluate
from tqdm import tqdm
import pandas as pd
import os
import gc

# ---------------------------------------------------------
# SETUP
# ---------------------------------------------------------
base_exp_path = f"{experiment_path}/{my_experiment}"
runs_to_eval = ["Baseline", "1", "2", "3", "4"]
final_results = []

BATCH_SIZE = 16

# Load Tokenizer once
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # <--- CRITICAL for batch generation

eval_subset = eval_dataset
references = [ex['output'] for ex in eval_subset]

# Pre-format all prompts to save time inside the loop
formatted_prompts = []
for example in eval_subset:
    user_input = example.get('input', "") or ""
    full_content = f"{example['instruction']}\n{user_input}".strip()
    prompt = f"<|im_start|>user\n{full_content}<|im_end|>\n<|im_start|>assistant\n"
    formatted_prompts.append(prompt)

# ---------------------------------------------------------
# EVALUATION LOOP (With BERTScore)
# ---------------------------------------------------------
# Load metrics *once* before the loop to save time
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

for run_id in runs_to_eval:
    print(f"\n" + "="*50)
    print(f"PROCESSING RUN: {run_id}")
    print("="*50)

    # 1. Clean Memory
    if 'model' in globals(): del model
    if 'base_model' in globals(): del base_model
    torch.cuda.empty_cache()
    gc.collect()

    # 2. Load Base Model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    # 3. Apply Adapter
    if run_id == "Baseline":
        model = base_model
    else:
        # Construct path (Adjust based on your folder structure)
        adapter_path = f"{base_exp_path}/runs/{run_id}/checkpoints/final_checkpoint"

        # Fallback check
        if not os.path.exists(adapter_path):
             chk_dir = f"{base_exp_path}/runs/{run_id}/checkpoints"
             if os.path.exists(chk_dir):
                 subdirs = [d for d in os.listdir(chk_dir) if d.startswith("checkpoint")]
                 if subdirs:
                     latest = sorted(subdirs, key=lambda x: int(x.split('-')[-1]))[-1]
                     adapter_path = f"{chk_dir}/{latest}"

        if not os.path.exists(adapter_path):
            print(f"⚠️  Adapter not found at {adapter_path}. Skipping.")
            continue

        print(f"   Loading Adapter from: {adapter_path}")
        model = PeftModel.from_pretrained(base_model, adapter_path)

    model.eval()

    # 4. Batched Generation
    print(f"   Generating answers (Batch Size: {BATCH_SIZE})...")
    predictions = []

    for i in tqdm(range(0, len(formatted_prompts), BATCH_SIZE)):
        batch_prompts = formatted_prompts[i : i + BATCH_SIZE]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=128, temperature=0.6, do_sample=True, pad_token_id=tokenizer.pad_token_id
            )

        input_len = inputs.input_ids.shape[1]
        batch_responses = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        predictions.extend(batch_responses)

    # 5. Metrics Calculation
    print("Calculating Metrics (ROUGE + BERTScore)...")

    # ROUGE
    r_scores = rouge.compute(predictions=predictions, references=references, rouge_types=["rougeL"])

    # BERTScore (Runs on CPU to avoid OOM)
    b_scores = bertscore.compute(
        predictions=predictions,
        references=references,
        lang="en",
        model_type="distilbert-base-uncased",
        device="cpu"  # Keep this CPU!
    )
    import numpy as np
    avg_bert = np.mean(b_scores['f1'])

    stats = {
        "Run": run_id,
        "ROUGE-L": round(r_scores['rougeL'], 4),
        "BERTScore": round(avg_bert, 4),
        "Sample Answer": predictions[0][:150] + "..."
    }
    final_results.append(stats)
    print(f"  ROUGE: {stats['ROUGE-L']} | BERT: {stats['BERTScore']}")

# ---------------------------------------------------------
# FINAL SUMMARY TABLE
# ---------------------------------------------------------
print("\n" + "="*60)
print("🏆 FINAL COMPARISON REPORT")
print("="*60)
df = pd.DataFrame(final_results)
cols = ["Run", "ROUGE-L", "BERTScore", "Sample Answer"]
print(df[cols].to_markdown(index=False))

## Conclusion

In this tutorial, we fine-tuned a cybersecurity Q&A assistant by running **4 SFT configurations concurrently** on a single GPU using RapidFire AI’s chunk-based scheduling. Here’s what we covered:

1. **Defined a Config Group** using `List()` wrappers and `RFGridSearch` to specify multiple LoRA adapters and training strategies.
2. **Ran all configurations concurrently** with `run_fit()`, getting comparative metrics after each data chunk.
3. **Monitored training in real time** via TensorBoard, comparing loss curves and eval metrics side-by-side.
4. **Evaluated all runs post-training** against the baseline using ROUGE-L and BERTScore, producing a final comparison report.

### Next Steps

- **Try different models**: Replace Qwen2.5-1.5B with larger models like [Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) or [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct).
- **Expand the grid**: Add more learning rates, LoRA ranks, or target module combinations.
- **Increase training steps**: Raise `max_steps` beyond 100 for deeper fine-tuning.
- **Explore other training methods**: Use `RFDPOConfig` or `RFGRPOConfig` for preference alignment (DPO/GRPO) instead of SFT.
- **Use the RapidFire UI**: For a richer monitoring experience beyond TensorBoard, run RapidFire locally with the full dashboard at `http://localhost:8853`.

For more details, see the [full SFT blog post](https://huggingface.co/blog/rapidfire-ai-inc/rapidfireai-sft-tutorial) and the [RapidFire AI documentation](https://oss-docs.rapidfire.ai).